# Thực nghiệm Retrieval: BM25, Dense, Hybrid và Reranking

Notebook 04 thay đổi cách biểu diễn/model embedding. Notebook này giữ E5 và
corpus cố định để quan sát cách tìm và xếp hạng đoạn văn.

1. BM25: tìm theo từ/thuật ngữ.
2. Dense: tìm bằng vector E5 trong Qdrant đã lưu ở notebook 03.
3. Hybrid: kết hợp hai danh sách bằng RRF.
4. Reranking: chấm lại ứng viên dense/hybrid bằng cross-encoder.

Chưa sinh câu trả lời bằng LLM, chưa làm app hay class Retrieval.
Chọn kernel .venv, chạy từ trên xuống. Cần chạy notebook 03 trước để có index.
Dependency BM25: python -m pip install rank-bm25==0.2.2.
Reranking được tắt mặc định để lượt đầu chạy nhanh; bật RUN_RERANK khi muốn
thực nghiệm đủ năm cấu hình. Lần đầu bật cần tải thêm model từ Hugging Face.

Đọc lý thuyết: [Retrieval Materials](../docs/materials/RetrievalMaterials.md).

## Import công cụ dữ liệu

Dùng JSON để đọc dữ liệu, NumPy để tính điểm và pandas để xem bảng.

In [1]:
import hashlib
import json
import re
import sys
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd

## Import công cụ Retrieval

BM25 dùng lexical ranking; Qdrant tìm vector. E5Embedder đã có trong src.

In [2]:
import torch
from rank_bm25 import BM25Okapi
from qdrant_client import QdrantClient, models

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.ingestion.textbook_embedder import E5Embedder

## Đường dẫn dữ liệu

Đọc đúng chunk và database đã dùng ở notebook 03; không parse lại PDF.

In [3]:
CHUNK_PATH = ROOT / "output/chunks/structure_aware_recursive.jsonl"
PARSER_PATH = ROOT / "output/parsed_chapter1.jsonl"
DB_PATH = ROOT / "output/qdrant_db"
PREVIOUS_RUN = ROOT / "output/embedding_comparison/structure_aware_recursive/manifest.json"
OUTPUT_PATH = ROOT / "output/retrieval/structure_aware_recursive"

## Cấu hình thực nghiệm

Giữ E5 cố định. Candidate là số ứng viên; top-k là số kết quả cuối.

In [4]:
MODEL_NAME = "intfloat/multilingual-e5-small"
MODEL_REVISION = "614241f622f53c4eeff9890bdc4f31cfecc418b3"
TOP_K = 5
CANDIDATE_K = 20
RRF_CONSTANT = 60
RUN_RERANK = False
torch.set_num_threads(4)
torch.manual_seed(42)

## Đọc chunk và Parser

Giữ nguyên text, metadata và thứ tự dòng để khớp point ID của notebook 03.

In [5]:
rows = [json.loads(line) for line in CHUNK_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
records = [json.loads(line) for line in PARSER_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
records_by_id = {record["id"]: record for record in records}
indices = [i for i, row in enumerate(rows) if row["metadata"]["sources"]]
chunks = [rows[i] for i in indices]
texts = [chunk["page_content"] for chunk in chunks]
print("Tổng chunk:", len(rows), "| Có nguồn:", len(chunks))

Tổng chunk: 241 | Có nguồn: 233


## Kiểm tra nguồn

Mỗi đoạn text trong sources phải khớp với record gốc. Heading không có nguồn được bỏ khỏi corpus tìm kiếm.

In [6]:
assert chunks and all(text.strip() for text in texts)
for chunk in chunks:
    for source in chunk["metadata"]["sources"]:
        record = records_by_id[source["record_id"]]
        original = record["text"][source["source_start_index"]:source["source_end_index"]]
        copied = chunk["page_content"][source["chunk_start_index"]:source["chunk_end_index"]]
        assert original == copied
print("Nguồn text: OK")

Nguồn text: OK


## Mở collection đã lưu

Dùng đúng cách tạo tên ở notebook 03. Chỉ đọc/search, không ghi lại database. Qdrant Local tìm exact; đây không phải benchmark HNSW.

In [7]:
key = MODEL_NAME + MODEL_REVISION + CHUNK_PATH.read_text(encoding="utf-8")
COLLECTION_NAME = "e5_" + hashlib.sha256(key.encode()).hexdigest()[:12]
assert DB_PATH.exists(), "Chạy notebook 03 trước để tạo database."
if "client" in globals():
    client.close()
client = QdrantClient(path=str(DB_PATH))
assert client.collection_exists(COLLECTION_NAME), "Index không khớp input/model; chạy notebook 03."
COLLECTION_NAME

'e5_183a932fed56'

## Đọc vector và payload

Đọc theo ID để lấy vector đối chứng. Đây chưa phải tìm kiếm bằng câu hỏi.

In [8]:
config = client.get_collection(COLLECTION_NAME).config.params.vectors
assert config.size == 384 and config.distance == models.Distance.COSINE
assert client.count(COLLECTION_NAME, exact=True).count == len(chunks)
points = client.retrieve(COLLECTION_NAME, ids=indices, with_payload=True, with_vectors=True)
points_by_id = {point.id: point for point in points}
assert set(points_by_id) == set(indices)

## Đối chiếu index với corpus

Kiểm tra toàn bộ payload. Vectors ở đây lấy chính từ index, không encode lại sách.

In [9]:
for point_id, chunk in zip(indices, chunks):
    assert points_by_id[point_id].payload == chunk
document_vectors = np.array([points_by_id[i].vector for i in indices], dtype=np.float32)
assert document_vectors.shape == (len(chunks), 384)
assert np.isfinite(document_vectors).all()
assert np.allclose(np.linalg.norm(document_vectors, axis=1), 1, atol=1e-5)
position_by_id = {point_id: position for position, point_id in enumerate(indices)}
document_vectors.shape

(233, 384)

# Bộ câu hỏi và bằng chứng

Giữ 8 chủ đề cũ (16 câu Anh–Việt) làm dev, thêm 8 chủ đề làm eval.
Hai bản dịch cùng chủ đề nằm cùng split. Đây là pilot nhỏ, không phải benchmark
độc lập quy mô lớn. Gold là một khoảng bằng chứng chọn trước; đoạn khác trả lời
đúng có thể chưa được gán. Không gọi Hit@k là Recall@k.

## Dùng lại câu hỏi notebook 04

Manifest cũ chứa câu hỏi và bằng chứng. Kiểm tra hash để không dùng nhầm corpus.

In [10]:
previous = json.loads(PREVIOUS_RUN.read_text(encoding="utf-8"))
assert previous["input_sha256"] == hashlib.sha256(CHUNK_PATH.read_bytes()).hexdigest()
assert previous["parser_sha256"] == hashlib.sha256(PARSER_PATH.read_bytes()).hexdigest()
cases = [dict(case, split="dev") for case in previous["cases"]]

## Câu hỏi eval: metadata

Gán một câu Anh, một câu Việt và đoạn bằng chứng đã kiểm tra trong Parser.

In [11]:
cases.append({
    "topic": "metadata",
    "record_suffix": "p40:b10",
    "en": "What is the information in the DBMS catalog called?",
    "vi": "Thông tin lưu trong catalog của DBMS được gọi là gì?",
    "evidence": "The information stored in the catalog is called meta-data, and it describes the structure of the primary database (Figure 1.1).",
    "split": "eval"
})

## Câu hỏi eval: abstraction

Gán một câu Anh, một câu Việt và đoạn bằng chứng đã kiểm tra trong Parser.

In [12]:
cases.append({
    "topic": "abstraction",
    "record_suffix": "p42:b5",
    "en": "Which characteristic supports program-data and program-operation independence?",
    "vi": "Đặc tính nào hỗ trợ độc lập chương trình–dữ liệu và chương trình–thao tác?",
    "evidence": "The characteristic that allows program-data independence and program-operation independence is called data abstraction.",
    "split": "eval"
})

## Câu hỏi eval: dba

Gán một câu Anh, một câu Việt và đoạn bằng chứng đã kiểm tra trong Parser.

In [13]:
cases.append({
    "topic": "dba",
    "record_suffix": "p45:b4",
    "en": "Who authorizes database access and monitors database use?",
    "vi": "Ai cấp quyền truy cập và giám sát việc sử dụng cơ sở dữ liệu?",
    "evidence": "The DBA is responsible for authorizing access to the database, coordinating and monitoring its use, and acquiring software and hardware resources as needed.",
    "split": "eval"
})

## Câu hỏi eval: transaction

Gán một câu Anh, một câu Việt và đoạn bằng chứng đã kiểm tra trong Parser.

In [14]:
cases.append({
    "topic": "transaction",
    "record_suffix": "p44:b2",
    "en": "What is a transaction in a database system?",
    "vi": "Giao dịch trong hệ thống cơ sở dữ liệu là gì?",
    "evidence": "A transaction is an executing program or process that includes one or more database accesses, such as reading or updating of database records.",
    "split": "eval"
})

## Câu hỏi eval: isolation

Gán một câu Anh, một câu Việt và đoạn bằng chứng đã kiểm tra trong Parser.

In [15]:
cases.append({
    "topic": "isolation",
    "record_suffix": "p44:b2",
    "en": "Which property makes a transaction appear to execute alone despite concurrent transactions?",
    "vi": "Thuộc tính nào khiến giao dịch có vẻ thực thi riêng dù nhiều giao dịch chạy đồng thời?",
    "evidence": "The isolation property ensures that each transaction appears to execute in isolation from other transactions, even though hundreds of transactions may be executing concurrently.",
    "split": "eval"
})

## Câu hỏi eval: atomicity

Gán một câu Anh, một câu Việt và đoạn bằng chứng đã kiểm tra trong Parser.

In [16]:
cases.append({
    "topic": "atomicity",
    "record_suffix": "p44:b2",
    "en": "Which property ensures all operations in a transaction happen or none happen?",
    "vi": "Thuộc tính nào đảm bảo tất cả thao tác của giao dịch được thực hiện hoặc không thao tác nào được thực hiện?",
    "evidence": "The atomicity property ensures that either all the database operations in a transaction are executed or none are.",
    "split": "eval"
})

## Câu hỏi eval: composite

Gán một câu Anh, một câu Việt và đoạn bằng chứng đã kiểm tra trong Parser.

In [17]:
cases.append({
    "topic": "composite",
    "record_suffix": "p95:b2",
    "en": "Can a composite attribute be divided into smaller meaningful parts?",
    "vi": "Thuộc tính phức hợp có thể chia thành các phần nhỏ có ý nghĩa riêng không?",
    "evidence": "Composite attributes can be divided into smaller subparts, which represent more basic attributes with independent meanings.",
    "split": "eval"
})

## Câu hỏi eval: weak_entity

Gán một câu Anh, một câu Việt và đoạn bằng chứng đã kiểm tra trong Parser.

In [18]:
cases.append({
    "topic": "weak_entity",
    "record_suffix": "p99:b1",
    "en": "What is an entity type without a key called?",
    "vi": "Kiểu thực thể không có khóa được gọi là gì?",
    "evidence": "An entity type may also have no key, in which case it is called a weak entity type (see Section 3.5).",
    "split": "eval"
})

## Xác định khoảng bằng chứng

Gold dùng record và offset, không phụ thuộc ID chunk. Evidence phải xuất hiện đúng một lần trong record.

In [19]:
for case in cases:
    matches = [record for record in records if record["id"].endswith(":" + case["record_suffix"])]
    assert len(matches) == 1
    record = matches[0]
    assert record["text"].count(case["evidence"]) == 1
    case["record_id"] = record["id"]
    case["start"] = record["text"].index(case["evidence"])
    case["end"] = case["start"] + len(case["evidence"])

## Ghép câu hỏi Anh–Việt

Mỗi query giữ cùng gold của chủ đề tương ứng.

In [20]:
queries = [dict(case, language=language, text=case[language])
           for case in cases for language in ["en", "vi"]]
pd.DataFrame(queries)[["topic", "split", "language", "text"]]

,topic,split,language,text
0,dbms,dev,en,What is a database management system?
1,dbms,dev,vi,Hệ quản trị cơ sở dữ liệu là gì?
2,schema,dev,en,What is a database schema?
3,schema,dev,vi,Lược đồ cơ sở dữ liệu là gì?
4,state,dev,en,What do we call the data in a database at a pa...
5,state,dev,vi,Dữ liệu trong cơ sở dữ liệu tại một thời điểm ...
6,logical,dev,en,Which data independence allows structural and ...
7,logical,dev,vi,Loại độc lập dữ liệu nào cho phép thay đổi cấu...
8,physical,dev,en,Why can physical files be reorganized without ...
9,physical,dev,vi,Vì sao có thể tổ chức lại các tệp vật lý mà kh...


## Hàm kiểm tra evidence

Một chunk khớp khi một source chứa trọn khoảng bằng chứng đã gán.

In [21]:
def contains_evidence(chunk, query):
    return any(source["record_id"] == query["record_id"]
               and source["source_start_index"] <= query["start"]
               and source["source_end_index"] >= query["end"]
               for source in chunk["metadata"]["sources"])

## Kiểm tra coverage gold

Không bỏ qua query có evidence bị chunking cắt mất; dừng để kiểm tra dữ liệu.

In [22]:
gold = np.array([[contains_evidence(chunk, query) for chunk in chunks] for query in queries])
assert gold.any(axis=1).all(), "Có evidence chưa được một chunk chứa trọn."
pd.Series(gold.sum(axis=1), name="gold_chunks_per_query").describe()

count    32.0
mean      1.0
std       0.0
min       1.0
25%       1.0
50%       1.0
75%       1.0
max       1.0
Name: gold_chunks_per_query, dtype: float64

# Thực nghiệm 1 — BM25

BM25 chấm các term khớp, có điều chỉnh tần suất và độ dài đoạn. Không cần model embedding.

## Tokenize

Dùng cùng quy tắc cho corpus và query: lowercase rồi lấy từ Unicode. Không dịch hoặc thêm từ khóa.

In [23]:
def tokenize(text):
    return re.findall(r"\w+", text.lower())

tokenize("What is a database schema?")

['what', 'is', 'a', 'database', 'schema']

## Tạo BM25

Fit thống kê một lần trên toàn corpus. Giữ tham số mặc định rõ ràng để dễ tái lập.

In [24]:
start = perf_counter()
bm25 = BM25Okapi([tokenize(text) for text in texts], k1=1.5, b=0.75, epsilon=0.25)
bm25_load_s = perf_counter() - start

## Lọc tập hợp lệ

Chapter và section được kiểm tra trước lấy top-k. Dùng section_ids để không bỏ chunk giao nhiều section.

In [25]:
def allowed_positions(chapter=None, section=None):
    return [i for i, chunk in enumerate(chunks)
            if (chapter is None or chunk["metadata"]["chapter_number"] == chapter)
            and (section is None or section in chunk["metadata"]["section_ids"])]

## Tìm kiếm BM25

Trả cặp (vị trí chunk, điểm), giảm dần theo điểm. Không bù kết quả điểm 0.

In [26]:
def bm25_search(query, limit, chapter=None, section=None):
    scores = bm25.get_scores(tokenize(query))
    eligible = allowed_positions(chapter, section)
    order = sorted(eligible, key=lambda i: (-scores[i], indices[i]))
    return [(i, float(scores[i])) for i in order[:limit] if scores[i] > 0]

bm25_search(queries[0]["text"], TOP_K)

[(187, 12.575295228432845),
 (188, 11.349691585602889),
 (8, 10.655664736782168),
 (167, 10.02852326312945),
 (2, 9.93945775527304)]

# Thực nghiệm 2 — E5 + Qdrant

Vector tài liệu đã có. Chỉ encode câu hỏi, rồi tìm các vector gần nhất.

## Load E5

E5Embedder thêm query prefix và kiểm tra giới hạn token. Model được load một lần.

In [27]:
start = perf_counter()
embedder = E5Embedder(model_name=MODEL_NAME, revision=MODEL_REVISION, batch_size=2)
e5_load_s = perf_counter() - start

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## Kiểm tra model với vector đã lưu

Encode một chunk mẫu để kiểm tra vector tương đương index trước khi query. Không embed lại cả corpus.

In [28]:
from langchain_core.documents import Document

sample_vector = embedder.embed_documents([Document(page_content=texts[0])])[0]
assert np.allclose(sample_vector, document_vectors[0], atol=1e-5)
print("E5 mẫu khớp index: OK")

E5 mẫu khớp index: OK


## Encode bộ câu hỏi

Giữ vector query để các phương pháp dùng cùng đầu vào. Thời gian encode được ghi riêng.

In [29]:
query_vectors = []
encode_ms = []
for query in queries:
    start = perf_counter()
    query_vectors.append(embedder.embed_query(query["text"]))
    encode_ms.append((perf_counter() - start) * 1000)
query_vectors = np.asarray(query_vectors)
query_vectors.shape

(32, 384)

## Tạo filter Qdrant

Hai điều kiện được ghép bằng AND. Giá trị chapter và section giữ đúng schema metadata.

In [30]:
def make_filter(chapter=None, section=None):
    conditions = []
    if chapter is not None:
        conditions.append(models.FieldCondition(key="metadata.chapter_number", match=models.MatchValue(value=chapter)))
    if section is not None:
        conditions.append(models.FieldCondition(key="metadata.section_ids", match=models.MatchValue(value=section)))
    return models.Filter(must=conditions) if conditions else None

## Tìm kiếm dense

query_points tìm theo vector; ID được ánh xạ về vị trí chunk để dùng cùng BM25.

In [31]:
def dense_search(vector, limit, chapter=None, section=None):
    response = client.query_points(
        collection_name=COLLECTION_NAME, query=vector.tolist(), limit=limit,
        query_filter=make_filter(chapter, section), with_payload=False,
    )
    hits = [(position_by_id[point.id], float(point.score)) for point in response.points]
    return sorted(hits, key=lambda hit: (-hit[1], indices[hit[0]]))

dense_search(query_vectors[0], TOP_K)

[(8, 0.9190409253109634),
 (169, 0.883808579564996),
 (9, 0.8704353981456114),
 (39, 0.869881734589085),
 (125, 0.8667202923261373)]

## Đối chứng Qdrant với NumPy

Cosine bằng dot product vì vectors đã chuẩn hóa. Xét score tại cutoff thay vì buộc cùng thứ tự khi có tie.

In [32]:
for vector in query_vectors:
    scores = document_vectors @ vector
    hits = dense_search(vector, TOP_K)
    cutoff = np.sort(scores)[-TOP_K]
    for position, score in hits:
        assert np.isclose(score, scores[position], atol=1e-5)
        assert scores[position] >= cutoff - 1e-5
print("Dense Qdrant / NumPy: OK")

Dense Qdrant / NumPy: OK


# Thực nghiệm 3 — Hybrid bằng RRF

Kết hợp thứ hạng BM25 và dense; không cộng trực tiếp hai loại similarity score.

## Hàm RRF

Rank bắt đầu 1. Chunk ở cả hai nhánh được cộng đóng góp; mỗi chunk chỉ xuất hiện một lần sau fusion.

In [33]:
def fuse(bm25_hits, dense_hits, limit):
    scores = {}
    for hits in [bm25_hits, dense_hits]:
        for rank, (position, _) in enumerate(hits, start=1):
            scores[position] = scores.get(position, 0) + 1 / (RRF_CONSTANT + rank)
    order = sorted(scores, key=lambda i: (-scores[i], indices[i]))
    return [(i, scores[i]) for i in order[:limit]]

## Ví dụ RRF tính tay

Chunk 0 nằm trong cả hai nhánh nên nhận hai đóng góp. Điểm gốc 9.0/0.8 không được cộng vào RRF.

In [34]:
example = fuse([(0, 9.0), (1, 5.0)], [(1, 0.8), (0, 0.7)], limit=2)
assert np.isclose(dict(example)[0], 1 / 61 + 1 / 62)
example

[(0, 0.03252247488101534), (1, 0.03252247488101534)]

## Khảo sát số ứng viên trên dev

Chỉ xem coverage trên 16 câu dev. Giữ candidate=20 cho lượt so sánh; không chọn tham số từ eval.

In [35]:
coverage = []
for limit in [10, 20, 40]:
    for q, query in enumerate(queries):
        if query["split"] != "dev":
            continue
        dense_hits = dense_search(query_vectors[q], limit)
        hybrid_hits = fuse(bm25_search(query["text"], limit), dense_hits, limit)
        for name, hits in [("dense", dense_hits), ("hybrid", hybrid_hits)]:
            coverage.append({"method": name, "candidate_k": limit,
                             "hit": int(any(gold[q, i] for i, _ in hits))})
pd.DataFrame(coverage).groupby(["method", "candidate_k"])["hit"].mean()

method  candidate_k
dense   10             0.8125
        20             0.9375
        40             0.9375
hybrid  10             0.8125
        20             0.9375
        40             0.9375
Name: hit, dtype: float64

## Lấy ứng viên cho bộ câu hỏi

Đo từng nhánh riêng, rồi fusion. Giữ danh sách này để reranker chấm đúng cùng ứng viên.

In [36]:
rankings = {name: [] for name in ["bm25", "dense", "hybrid"]}
stage_times = []
for q, query in enumerate(queries):
    start = perf_counter()
    lexical = bm25_search(query["text"], CANDIDATE_K)
    lexical_ms = (perf_counter() - start) * 1000
    start = perf_counter()
    dense = dense_search(query_vectors[q], CANDIDATE_K)
    dense_ms = (perf_counter() - start) * 1000
    start = perf_counter()
    hybrid = fuse(lexical, dense, CANDIDATE_K)
    fusion_ms = (perf_counter() - start) * 1000
    for name, hits in [("bm25", lexical), ("dense", dense), ("hybrid", hybrid)]:
        rankings[name].append(hits)
    stage_times.append({"query_index": q, "encode_ms": encode_ms[q],
                        "bm25_ms": lexical_ms, "dense_ms": dense_ms, "fusion_ms": fusion_ms})

## Xem top-k và nguồn

Một hàm nhỏ để xem kết quả của mọi phương pháp. Số trang hiển thị là PDF index + 1.

In [37]:
def top_results(method, query_index=0, k=TOP_K):
    result = []
    for rank, (position, score) in enumerate(rankings[method][query_index][:k], 1):
        metadata = chunks[position]["metadata"]
        result.append({"rank": rank, "point_id": indices[position], "score": score,
                       "evidence_match": bool(gold[query_index, position]),
                       "chapter": metadata["chapter_number"], "section": metadata["section_number"],
                       "pdf_pages": sorted({s["pdf_page"] + 1 for s in metadata["sources"]}),
                       "preview": texts[position][:220]})
    return pd.DataFrame(result)

top_results("hybrid", query_index=1)

,rank,point_id,score,evidence_match,chapter,section,pdf_pages,preview
0,1,8,0.016393,True,1,1.1,[37],A database management system (DBMS) is a compu...
1,2,146,0.016129,False,2,2.4.1,"[73, 74]",Let us consider the top part of Figure 2.3 fir...
2,3,202,0.015873,False,2,2.7,[90],"In this chapter, we follow the traditional app..."
3,4,130,0.015625,False,2,2.3.1,[70],In DBMSs where a clear separation is maintaine...
4,5,40,0.015385,False,1,1.4.1,[46],[[SECTION 1.4.1: Database Administrators]]\nIn...


# Thực nghiệm 4 — Reranking (bật riêng)

Bật RUN_RERANK=True ở Setup rồi chạy từ trên xuống để thêm dense_rerank và
hybrid_rerank. BGE reranker đọc từng cặp query–chunk, không dùng vector E5
để chấm lại. Lần đầu cần tải model; CPU có thể chạy lâu.
Không có kết quả reranking khi công tắc tắt; không dùng điểm giả để thay thế.

## Load reranker

Load model đa ngôn ngữ một lần nếu bật. Khi tải được model, ghi lại revision thực tế.

In [38]:
RERANKER_NAME = "BAAI/bge-reranker-v2-m3"
reranker_revision = None
reranker_load_s = None
if RUN_RERANK:
    from sentence_transformers import CrossEncoder
    start = perf_counter()
    reranker = CrossEncoder(RERANKER_NAME, device="cpu", max_length=512)
    reranker_revision = getattr(reranker.model.config, "_commit_hash", None)
    reranker_load_s = perf_counter() - start
else:
    print("Chưa chạy reranking. Bật RUN_RERANK khi cần.")

Chưa chạy reranking. Bật RUN_RERANK khi cần.


## Chấm lại một danh sách ứng viên

Đếm token của cặp input để không truncate ngầm. Raw score chỉ dùng so thứ hạng trong cùng query.

In [39]:
def rerank(query, hits):
    if not hits:
        return []
    pairs = [[query, texts[position]] for position, _ in hits]
    for question, passage in pairs:
        count = len(reranker.tokenizer(question, passage, truncation=False)["input_ids"])
        assert count <= 512, f"Cặp query–chunk vượt giới hạn: {count} token."
    scores = np.asarray(reranker.predict(pairs, batch_size=2, show_progress_bar=False,
                                        activation_fn=torch.nn.Identity())).reshape(-1)
    assert len(scores) == len(hits) and np.isfinite(scores).all()
    result = [(position, float(score)) for (position, _), score in zip(hits, scores)]
    return sorted(result, key=lambda hit: (-hit[1], indices[hit[0]]))

## Smoke test reranker

Kiểm tra hai cặp trước khi chạy toàn bộ ứng viên. Tập ứng viên và text nguồn không thay đổi.

In [40]:
if RUN_RERANK:
    sample_hits = rankings["dense"][0][:2]
    reranked_sample = rerank(queries[0]["text"], sample_hits)
    assert {i for i, _ in sample_hits} == {i for i, _ in reranked_sample}
    print(reranked_sample)

## Rerank dense và hybrid

Chỉ chấm ứng viên đã lấy. Reranker không thể thêm bằng chứng bị thiếu ở bước lấy ứng viên.

In [41]:
if RUN_RERANK:
    for base in ["dense", "hybrid"]:
        method = base + "_rerank"
        rankings[method] = []
        for q, query in enumerate(queries):
            start = perf_counter()
            hits = rerank(query["text"], rankings[base][q])
            stage_times[q][method + "_ms"] = (perf_counter() - start) * 1000
            assert {i for i, _ in hits} == {i for i, _ in rankings[base][q]}
            rankings[method].append(hits)
        print("Hoàn tất:", method)

# Đánh giá kết quả

Hit@k: có evidence trong top-k. MRR@k: nghịch đảo hạng đầu tiên có evidence; miss bằng 0. Báo riêng ngôn ngữ và split.

## Hàm tính Hit và MRR

Chấm đúng danh sách đã xếp hạng, không tìm thêm chunk để sửa kết quả.

In [42]:
def evaluate(method, k):
    result = []
    for q, query in enumerate(queries):
        matched = [rank for rank, (i, _) in enumerate(rankings[method][q][:k], 1) if gold[q, i]]
        result.append({"method": method, "k": k, "query_index": q, "topic": query["topic"],
                       "split": query["split"], "language": query["language"], "query": query["text"],
                       "hit": int(bool(matched)), "mrr": 1 / matched[0] if matched else 0.0})
    return pd.DataFrame(result)

## Bảng chất lượng

Dùng cùng câu hỏi và corpus cho mọi phương pháp. Không so trị tuyệt đối của các loại score khác nhau.

In [43]:
per_query = pd.concat([evaluate(method, k) for method in rankings for k in [1, 3, 5, 10]], ignore_index=True)
summary = per_query.groupby(["split", "language", "method", "k"])[["hit", "mrr"]].mean()
summary.xs(TOP_K, level="k").round(3)

hit    mrr
split language method              
dev   en       bm25    0.875  0.729
               dense   1.000  0.938
               hybrid  1.000  1.000
      vi       bm25    0.000  0.000
               dense   0.625  0.500
               hybrid  0.625  0.500
eval  en       bm25    1.000  0.754
               dense   1.000  0.900
               hybrid  1.000  0.938
      vi       bm25    0.000  0.000
               dense   0.500  0.344
               hybrid  0.500  0.344

## Coverage trước reranking

Nếu candidate không chứa evidence, reranker không thể cứu query đó. Bảng này dùng candidate=20.

In [44]:
candidate_summary = pd.concat([evaluate(method, CANDIDATE_K) for method in ["dense", "hybrid"]])
candidate_summary.groupby(["split", "language", "method"])["hit"].mean()

split  language  method
dev    en        dense     1.000
                 hybrid    1.000
       vi        dense     0.875
                 hybrid    0.875
eval   en        dense     1.000
                 hybrid    1.000
       vi        dense     0.875
                 hybrid    0.875
Name: hit, dtype: float64

## Xem các câu miss

Kiểm tra từng query thay vì chỉ kết luận từ trung bình trên pilot nhỏ.

In [45]:
per_query.loc[(per_query["k"] == TOP_K) & (per_query["hit"] == 0),
              ["method", "split", "language", "topic", "query"]]

,method,split,language,topic,query
65,bm25,dev,vi,dbms,Hệ quản trị cơ sở dữ liệu là gì?
66,bm25,dev,en,schema,What is a database schema?
67,bm25,dev,vi,schema,Lược đồ cơ sở dữ liệu là gì?
69,bm25,dev,vi,state,Dữ liệu trong cơ sở dữ liệu tại một thời điểm ...
71,bm25,dev,vi,logical,Loại độc lập dữ liệu nào cho phép thay đổi cấu...
73,bm25,dev,vi,physical,Vì sao có thể tổ chức lại các tệp vật lý mà kh...
75,bm25,dev,vi,multivalued,Một thuộc tính có thể có nhiều giá trị cho cùn...
77,bm25,dev,vi,derived,Vì sao tuổi có thể tính từ ngày sinh thay vì l...
79,bm25,dev,vi,key,Thuộc tính nào có thể xác định duy nhất từng t...
81,bm25,eval,vi,metadata,Thông tin lưu trong catalog của DBMS được gọi ...


## So top-k của cùng một câu

Đổi QUERY_INDEX để xem nguyên nhân lexical/dense/hybrid xếp hạng khác nhau.

In [46]:
QUERY_INDEX = 1
print(queries[QUERY_INDEX]["text"])
comparison = pd.concat({method: top_results(method, QUERY_INDEX) for method in rankings})
comparison

Hệ quản trị cơ sở dữ liệu là gì?


rank  point_id     score evidence_match chapter section pdf_pages  \
dense  0   1.0       8.0  0.803873           True       1     1.1      [37]   
       1   2.0     146.0  0.798343          False       2   2.4.1  [73, 74]   
       2   3.0     202.0  0.796472          False       2     2.7      [90]   
       3   4.0     130.0  0.796358          False       2   2.3.1      [70]   
       4   5.0      40.0  0.796032          False       1   1.4.1      [46]   
hybrid 0   1.0       8.0  0.016393           True       1     1.1      [37]   
       1   2.0     146.0  0.016129          False       2   2.4.1  [73, 74]   
       2   3.0     202.0  0.015873          False       2     2.7      [90]   
       3   4.0     130.0  0.015625          False       2   2.3.1      [70]   
       4   5.0      40.0  0.015385          False       1   1.4.1      [46]   

                                                    preview  
dense  0  A database management system (DBMS) is a compu...  
       1  Let us consider the top part of Figure 2.3 fir...  
       2  In this chapter, we follow the traditional app...  
       3  In DBMSs where a clear separation is maintaine...  
       4  [[SECTION 1.4.1: Database Administrators]]\nIn...  
hybrid 0  A database management system (DBMS) is a compu...  
       1  Let us consider the top part of Figure 2.3 fir...  
       2  In this chapter, we follow the traditional app...  
       3  In DBMSs where a clear separation is maintaine...  
       4  [[SECTION 1.4.1: Database Administrators]]\nIn...

## Đọc đoạn đầy đủ

Xem text gốc của một kết quả và các record nguồn, tránh đánh giá chỉ từ preview.

In [47]:
METHOD = "dense"
position = rankings[METHOD][QUERY_INDEX][0][0]
print(texts[position])
pd.DataFrame([{"record_id": s["record_id"], "pdf_page": s["pdf_page"] + 1,
               "text": records_by_id[s["record_id"]]["text"]}
              for s in chunks[position]["metadata"]["sources"]])

A database management system (DBMS) is a computerized system that enables users to create and maintain a database. The DBMS is a general-purpose software system that facilitates the processes of defining, constructing, manipulating, and sharing databases among various users and applications. Defining a database involves specifying the data types, structures, and constraints of the data to be stored in the database. The database definition or descriptive information is also stored by the DBMS in the form of a database catalog or dictionary; it is called meta-data. Constructing the database is the process of storing the data on some storage medium that is controlled by the DBMS. Manipulating a database includes functions such as querying the database to retrieve specific data, updating the database to reflect changes in the miniworld, and generating reports from the data. Sharing a database allows multiple users and programs to access the database simultaneously.


,record_id,pdf_page,text
0,863264e73f7a57ea:p36:b2,37,A database management system (DBMS) is a compu...


## Thời gian từng bước

Đo một lượt trên máy hiện tại; encode được dùng lại giữa các phương pháp. Đây là thời gian quan sát, chưa phải benchmark latency/SLA.

In [48]:
timing = pd.DataFrame(stage_times)
timing.drop(columns="query_index").median().round(3).rename("median_ms")

encode_ms    27.595
bm25_ms       0.608
dense_ms      0.695
fusion_ms     0.017
Name: median_ms, dtype: float64

# Kiểm tra filter và query ngoài corpus

Các ca này kiểm tra hành vi; không cộng vào bảng Hit/MRR của câu hỏi có gold.

## Hàm gọi ba phương pháp cơ bản

Một query được encode khi dùng dense/hybrid. Hàm nhỏ này phục vụ thử filter và nhập câu hỏi khác.

In [49]:
def search(query, method="dense", limit=TOP_K, chapter=None, section=None):
    assert isinstance(query, str) and query.strip(), "Query phải là chuỗi không rỗng."
    assert type(limit) is int and limit > 0
    assert method in ["bm25", "dense", "hybrid"]
    if method == "bm25":
        return bm25_search(query, limit, chapter, section)
    vector = embedder.embed_query(query)
    dense = dense_search(vector, max(limit, CANDIDATE_K), chapter, section)
    if method == "dense":
        return dense[:limit]
    lexical = bm25_search(query, max(limit, CANDIDATE_K), chapter, section)
    return fuse(lexical, dense, limit)

## Kiểm tra bốn ca filter

Lọc trước khi ranking. Ca không có chapter phù hợp phải trả danh sách rỗng.

In [50]:
filter_cases = [{"chapter": "1"}, {"section": "sec_1.1"},
                {"chapter": "1", "section": "sec_1.1"}, {"chapter": "999"}]
for filters in filter_cases:
    eligible = set(allowed_positions(**filters))
    for method in ["bm25", "dense", "hybrid"]:
        hits = search("What is a database management system?", method, **filters)
        assert all(i in eligible for i, _ in hits)
        if not eligible:
            assert hits == []
print("BM25 / dense / hybrid filters: OK")

BM25 / dense / hybrid filters: OK


## Thử câu ngoài corpus

Dense vẫn trả đoạn gần nhất. Kết quả có score cao không bảo đảm câu hỏi được tài liệu trả lời.

In [51]:
outside_queries = ["How do I bake a chocolate cake?", "How can I repair a bicycle tire?",
                   "Làm thế nào để làm bánh sô-cô-la?", "Cách sửa lốp xe đạp bị thủng là gì?"]
outside_results = []
for query in outside_queries:
    for method in ["bm25", "dense", "hybrid"]:
        hits = search(query, method, limit=1)
        outside_results.append({"query": query, "method": method,
                                "score": hits[0][1] if hits else None,
                                "preview": texts[hits[0][0]][:160] if hits else None})
pd.DataFrame(outside_results)

,query,method,score,preview
0,How do I bake a chocolate cake?,bm25,7.491703,CS3380 CS3320\nCS3380 MATH2410\nCS3320 CS1310\...
1,How do I bake a chocolate cake?,dense,0.779147,2.2. Discuss the main categories of data model...
2,How do I bake a chocolate cake?,hybrid,0.032522,2.2. Discuss the main categories of data model...
3,How can I repair a bicycle tire?,bm25,7.491703,CS3380 CS3320\nCS3380 MATH2410\nCS3320 CS1310\...
4,How can I repair a bicycle tire?,dense,0.784726,2.2. Discuss the main categories of data model...
5,How can I repair a bicycle tire?,hybrid,0.029907,2.2. Discuss the main categories of data model...
6,Làm thế nào để làm bánh sô-cô-la?,bm25,NaN,NaN
7,Làm thế nào để làm bánh sô-cô-la?,dense,0.747988,architectures?\n2.10. Discuss some types of da...
8,Làm thế nào để làm bánh sô-cô-la?,hybrid,0.016393,architectures?\n2.10. Discuss some types of da...
9,Cách sửa lốp xe đạp bị thủng là gì?,bm25,NaN,NaN


# Lưu kết quả

Mỗi lần chạy lưu bảng và cấu hình thực tế. Chỉ ghi phương pháp đã chạy; không ghi reranking là thành công khi công tắc tắt.

## Tạo thư mục và lưu bảng

Kết quả nằm riêng trong output/retrieval, không ghi đè bảng embedding của notebook 04.

In [52]:
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
per_query.to_csv(OUTPUT_PATH / "per_query.csv", index=False, encoding="utf-8-sig")
summary.to_csv(OUTPUT_PATH / "summary.csv", encoding="utf-8-sig")
timing.to_csv(OUTPUT_PATH / "timing.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(coverage).to_csv(OUTPUT_PATH / "candidate_coverage.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(outside_results).to_csv(OUTPUT_PATH / "outside_queries.csv", index=False, encoding="utf-8-sig")

## Lưu query và top-k

Lưu đầy đủ text/metadata của kết quả để truy vết; vị trí chunk chỉ có ý nghĩa trong corpus này.

In [53]:
with (OUTPUT_PATH / "queries.jsonl").open("w", encoding="utf-8") as stream:
    for query in queries:
        stream.write(json.dumps(query, ensure_ascii=False) + "\n")
with (OUTPUT_PATH / "topk.jsonl").open("w", encoding="utf-8") as stream:
    for method, lists in rankings.items():
        for q, hits in enumerate(lists):
            for rank, (position, score) in enumerate(hits[:TOP_K], 1):
                item = dict(method=method, query_index=q, rank=rank, point_id=indices[position],
                            score=score, evidence_match=bool(gold[q, position]), **chunks[position])
                stream.write(json.dumps(item, ensure_ascii=False) + "\n")

## Manifest của lượt chạy

Ghi hash dữ liệu, model, tham số và phương pháp thực sự chạy để biết kết quả thuộc cấu hình nào.

In [54]:
from importlib.metadata import version

manifest = {"collection": COLLECTION_NAME, "chunks": len(chunks), "queries": len(queries),
            "chunk_sha256": hashlib.sha256(CHUNK_PATH.read_bytes()).hexdigest(),
            "parser_sha256": hashlib.sha256(PARSER_PATH.read_bytes()).hexdigest(),
            "model": MODEL_NAME, "model_revision": MODEL_REVISION,
            "candidate_k": CANDIDATE_K, "top_k": TOP_K, "rrf_constant": RRF_CONSTANT,
            "bm25": {"k1": 1.5, "b": 0.75, "epsilon": 0.25, "tokenizer": "Unicode words + lowercase"},
            "methods": list(rankings), "reranker": RERANKER_NAME if RUN_RERANK else None,
            "reranker_revision": reranker_revision, "reranker_max_length": 512,
            "load_seconds": {"bm25": bm25_load_s, "e5": e5_load_s, "reranker": reranker_load_s},
            "packages": {name: version(name) for name in ["rank-bm25", "qdrant-client", "sentence-transformers", "torch"]}}
(OUTPUT_PATH / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")

907

## Đóng database

Giải phóng Qdrant Local để notebook khác có thể mở lại cùng thư mục.

In [55]:
client.close()
print("Đã lưu kết quả:", OUTPUT_PATH)

Đã lưu kết quả: d:\Users\BuiTrongNguyen\CodexProject\db-rag-eval\output\retrieval\structure_aware_recursive


# Cách đọc kết quả

- So BM25 với dense để thấy lexical và semantic khác nhau; đọc riêng Anh/Việt.
- So hybrid với hai nhánh để xem fusion giúp hay đưa nhiễu lên.
- Nếu bật reranking, so với đúng phương pháp trước rerank và candidate coverage.
- Xem các câu miss và nguồn: lỗi có thể do vocabulary, model hoặc bằng chứng đã gán.
- Đọc dev để khảo sát; eval báo riêng. Hai tập nhỏ nên chưa kết luận phương pháp tốt nhất cho toàn sách.
- Giữ nguyên preprocessing/corpus khi so phương pháp; không dùng similarity làm độ tin cậy trả lời.

Sau khi hiểu kết quả, bước kế tiếp là tách các hàm search/fusion/rerank thành
class dùng chung; phần đó không triển khai trong notebook thực nghiệm này.

Nguồn: [BM25](https://nlp.stanford.edu/IR-book/html/htmledition/okapi-bm25-a-non-binary-model-1.html),
[Qdrant query](https://qdrant.tech/documentation/search/search/),
[RRF](https://cormack.uwaterloo.ca/cormacksigir09-rrf.pdf),
[BGE reranker](https://huggingface.co/BAAI/bge-reranker-v2-m3).